In [ ]:
# %% [markdown]
# # Gas Price Forecast - Hybrid Baseline (LightGBM + PyTorch)
# 
# - Reads `train.csv` & `test-template.csv`
# - Builds continuous date index (2020-01-02 → 2025-09-22)
# - Creates lag/date/rolling features
# - Trains LightGBM baseline and optional LSTM (PyTorch)
# - Writes `submission.csv`

# %%
import os, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TARGET_COL = "price_usd_per_mmbtu"
TRAIN_FN = "train.csv"
TEST_FN = "test-template.csv"
SUBMISSION_FN = "submission.csv"

PRED_START, PRED_END = "2020-01-02", "2025-09-22"

# %%
# --- Load and normalize ---
train = pd.read_csv(TRAIN_FN)
test = pd.read_csv(TEST_FN)

# detect date column
def find_date_col(df):
    for c in df.columns:
        if "date" in c.lower() or "id" == c.lower():
            return c
    return df.columns[0]

train_date_col = find_date_col(train)
test_date_col = find_date_col(test)

train["date"] = pd.to_datetime(train[train_date_col])
test["date"] = pd.to_datetime(test[test_date_col])
train = train.sort_values("date")

# %%
# --- Continuous date index ---
full_dates = pd.date_range(PRED_START, PRED_END, freq="D")
hist = train.set_index("date")[TARGET_COL].reindex(full_dates)
hist = hist.fillna(method="ffill", limit=2)
df = pd.DataFrame({"date": full_dates, TARGET_COL: hist.values})

# %%
# --- Features ---
def make_features(df):
    df = df.copy()
    df["dayofweek"] = df["date"].dt.dayofweek
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    for lag in [1,2,3,7,14]:
        df[f"lag_{lag}"] = df[TARGET_COL].shift(lag)
    df["roll_7"] = df[TARGET_COL].shift(1).rolling(7).mean()
    df["roll_14"] = df[TARGET_COL].shift(1).rolling(14).mean()
    return df

df = make_features(df)

# %%
# --- Train/validation split ---
train_mask = df["date"].isin(train["date"])
train_df = df.loc[train_mask].dropna().copy()

val_days = 90
cutoff = train_df["date"].max() - pd.Timedelta(days=val_days)
train_X = train_df[train_df["date"] <= cutoff]
val_X = train_df[train_df["date"] > cutoff]

features = [c for c in train_X.columns if c not in ["date", TARGET_COL]]

X_train, y_train = train_X[features].values, train_X[TARGET_COL].values
X_val, y_val = val_X[features].values, val_X[TARGET_COL].values

print(f"Train {X_train.shape}, Val {X_val.shape}")

# %%
# --- LightGBM baseline ---
try:
    import lightgbm as lgb
    lgb_train = lgb.Dataset(X_train, y_train)
    lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)
    params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "seed": RANDOM_STATE,
    }
    print("Training LightGBM...")
    model = lgb.train(params, lgb_train, valid_sets=[lgb_train, lgb_val],
                      num_boost_round=1000, early_stopping_rounds=50, verbose_eval=100)
except Exception as e:
    print("LightGBM unavailable:", e)
    from sklearn.ensemble import RandomForestRegressor
    model = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)

# %%
# --- Evaluate baseline ---
val_preds = model.predict(X_val)
rmse = mean_squared_error(y_val, val_preds, squared=False)
mae = mean_absolute_error(y_val, val_preds)
print(f"LightGBM RMSE: {rmse:.4f}, MAE: {mae:.4f}")

# %%
# --- Optional: PyTorch model ---
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Define simple dense model (fast, enough for baseline)
class SimpleRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x): return self.net(x)

# ---- define datasets AFTER X_train / y_train exist ----
train_dataset = TimeSeriesDataset(X_train, y_train)
val_dataset = TimeSeriesDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# Train small neural net
device = "cuda" if torch.cuda.is_available() else "cpu"
model_nn = SimpleRegressor(X_train.shape[1]).to(device)
opt = torch.optim.Adam(model_nn.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(20):
    model_nn.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device).view(-1,1)
        opt.zero_grad()
        pred = model_nn(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        opt.step()
    # Validation
    model_nn.eval()
    with torch.no_grad():
        val_loss = 0
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device).view(-1,1)
            val_loss += loss_fn(model_nn(xb), yb).item()
    print(f"Epoch {epoch+1:02d}, val_loss={val_loss/len(val_loader):.4f}")

# %%
# --- Predict for submission ---
test_dates = pd.date_range(PRED_START, PRED_END)
test_df = df[df["date"].isin(test_dates)].copy()
test_df = test_df.fillna(method="ffill")
X_test = test_df[features].values

# Combine LightGBM & NN predictions (weighted average)
pred_lgb = model.predict(X_test)
with torch.no_grad():
    pred_nn = model_nn(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy().ravel()

pred_final = 0.7 * pred_lgb + 0.3 * pred_nn

submission = pd.DataFrame({
    "id": test_df["date"].dt.strftime("%Y-%m-%d"),
    TARGET_COL: pred_final
})
submission.to_csv(SUBMISSION_FN, index=False, float_format="%.6f")
print(f"Saved {SUBMISSION_FN} ({len(submission)} rows)")
submission.head()



Building LSTM model with PyTorch...
Using device: cuda
GPU: NVIDIA GeForce RTX 3060


NameError: name 'X_train' is not defined